# 음식 인식 모델 held-out 평가

앱에 넣은 `yolov8n_food.tflite` 를 **학습에 쓰지 않은 사진**으로 평가한다.

학습은 AI Hub 74번의 **Validation 원천**으로만 했으므로 **Training 원천**은 모델이 한 번도 보지 않았다. 같은 342클래스의 미공개 사진이라 클래스별 정확도를 정직하게 잴 수 있다.

### 재는 것
- **Top-1 정확도**: 가장 높은 점수의 클래스가 정답인가
- **검출률(recall@0.40)**: 정답 클래스가 앱 임계값 0.40 을 넘었는가 → 넘지 못하면 앱에서 아예 안 보인다
- **클래스별 성적**과 **혼동 쌍**(무엇을 무엇으로 착각하는가)
- 정답이 임계값에 **근소 미달**인지(0.25~0.40) → 임계값을 낮출지 판단하는 근거

### 준비
1. PC 에서 `node tools/build_eval_set.js --images <원천 해제 폴더> --per-class 20` 실행 → `eval_set/` 생성
2. `eval_set` 을 tar 로 묶어 Drive `MyDrive/trex/eval_set.tar` 에 업로드
3. 평가할 모델 `yolov8n_food.tflite` 도 Drive `MyDrive/trex/` 에 올린다(앱 assets 의 것과 같은 파일)
4. 런타임은 GPU 가 아니어도 된다(CPU 로 충분). 아래 셀을 순서대로 실행.

In [ ]:
# 1. Drive 연결 + 평가셋·모델 준비
from google.colab import drive
drive.mount('/content/drive')

EVAL_TAR = '/content/drive/MyDrive/trex/eval_set.tar'
MODEL    = '/content/drive/MyDrive/trex/yolov8n_food.tflite'
THRESHOLD = 0.40   # 앱 FoodDetector 와 같은 값

import os
for p in (EVAL_TAR, MODEL):
    assert os.path.exists(p), f'{p} 가 없다'
!rm -rf /content/eval && mkdir -p /content/eval && tar -xf "{EVAL_TAR}" -C /content/eval
!echo "평가 이미지: $(ls /content/eval/images | wc -l)장" && head -3 /content/eval/ground_truth.csv

In [ ]:
# 2. 모델 로드 — 앱 FoodDetector 와 같은 전처리(letterbox + NHWC/NCHW 자동)를 쓴다
import numpy as np, tensorflow as tf
from PIL import Image

interp = tf.lite.Interpreter(model_path=MODEL, num_threads=4)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
shape = list(inp['shape'])
channels_first = shape[1] == 3 and shape[3] != 3
H, W = (shape[2], shape[3]) if channels_first else (shape[1], shape[2])
labels = [l.strip() for l in open('/content/eval/labels.txt', encoding='utf-8') if l.strip()]
print(f'입력 {shape} ({"NCHW" if channels_first else "NHWC"}), 출력 {list(out["shape"])}, 라벨 {len(labels)}종')

def preprocess(path):
    """앱과 동일: 비율 유지 축소 + 회색(114) 패딩, 0~1 정규화."""
    im = Image.open(path).convert('RGB')
    s = min(W / im.width, H / im.height)
    nw, nh = max(1, int(im.width * s)), max(1, int(im.height * s))
    canvas = Image.new('RGB', (W, H), (114, 114, 114))
    canvas.paste(im.resize((nw, nh), Image.BILINEAR), ((W - nw) // 2, (H - nh) // 2))
    a = np.asarray(canvas, dtype=np.float32) / 255.0
    a = a.transpose(2, 0, 1) if channels_first else a
    return a[None, ...]

def scores(path):
    """클래스별 최고 점수 — 앱 runInference 와 같은 집계."""
    interp.set_tensor(inp['index'], preprocess(path))
    interp.invoke()
    o = interp.get_tensor(out['index'])[0]
    o = o if o.shape[0] == len(labels) + 4 else o.T   # (4+nc, boxes) 로 맞춘다
    return o[4:, :].max(axis=1)

In [ ]:
# 3. 전체 추론 (이미지 수에 따라 수 분)
import csv, time
rows = list(csv.DictReader(open('/content/eval/ground_truth.csv', encoding='utf-8')))
results = []
t0 = time.time()
for i, r in enumerate(rows):
    s = scores('/content/eval/images/' + r['file'])
    gt = int(r['class_index'])
    order = np.argsort(-s)
    results.append({
        'file': r['file'], 'gt': gt, 'gt_name': r['class_name'],
        'gt_score': float(s[gt]),
        'top1': int(order[0]), 'top1_name': labels[order[0]], 'top1_score': float(s[order[0]]),
        'rank': int(np.where(order == gt)[0][0]),
    })
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(rows)}  ({(time.time()-t0)/(i+1)*1000:.0f}ms/장)')
print(f'완료 {len(results)}장, {time.time()-t0:.0f}초')

In [ ]:
# 4. 전체 지표
import pandas as pd
df = pd.DataFrame(results)
n = len(df)
top1 = (df.top1 == df.gt).mean()
top5 = (df['rank'] < 5).mean()
detected = (df.gt_score >= THRESHOLD).mean()
near = ((df.gt_score >= 0.25) & (df.gt_score < THRESHOLD)).mean()
print(f'이미지 {n}장 · 클래스 {df.gt.nunique()}종')
print(f'Top-1 정확도      {top1:.1%}')
print(f'Top-5 정확도      {top5:.1%}')
print(f'검출률(≥{THRESHOLD})   {detected:.1%}   ← 앱에서 실제로 보이는 비율')
print(f'근소 미달(0.25~{THRESHOLD}) {near:.1%}   ← 임계값을 낮추면 살아나는 비율')
print(f'정답 점수 중앙값   {df.gt_score.median():.2f}')

In [ ]:
# 5. 클래스별 성적 — 어떤 음식이 약한가
per = df.groupby('gt_name').agg(
    장수=('file', 'size'),
    top1정확도=('top1', lambda x: (x == df.loc[x.index, 'gt']).mean()),
    검출률=('gt_score', lambda x: (x >= THRESHOLD).mean()),
    점수중앙값=('gt_score', 'median'),
).sort_values('검출률')
print('=== 가장 약한 20종 ===')
display(per.head(20).style.format({'top1정확도': '{:.0%}', '검출률': '{:.0%}', '점수중앙값': '{:.2f}'}))
print('=== 가장 강한 10종 ===')
display(per.tail(10).style.format({'top1정확도': '{:.0%}', '검출률': '{:.0%}', '점수중앙값': '{:.2f}'}))

In [ ]:
# 6. 혼동 쌍 — 무엇을 무엇으로 착각하는가 (많은 순)
wrong = df[df.top1 != df.gt]
pairs = wrong.groupby(['gt_name', 'top1_name']).size().sort_values(ascending=False).head(25)
print(f'오분류 {len(wrong)}건 / {n}건')
display(pairs.rename('건수').to_frame())

In [ ]:
# 7. 임계값을 바꾸면 검출률이 어떻게 변하나 — 조정 판단 근거
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    d = (df.gt_score >= t).mean()
    # 오탐 대리 지표: 정답이 아닌데 그 문턱을 넘은 Top-1 의 비율
    fp = ((df.top1 != df.gt) & (df.top1_score >= t)).mean()
    mark = '  ← 현재' if abs(t - THRESHOLD) < 1e-9 else ''
    print(f'임계 {t:.2f}: 검출률 {d:.1%} · 오분류가 표시되는 비율 {fp:.1%}{mark}')

df.to_csv('/content/eval_results.csv', index=False)
per.to_csv('/content/eval_per_class.csv')
print('\n저장: /content/eval_results.csv, /content/eval_per_class.csv')
print('두 파일을 내려받아 docs 에 요약을 남긴다.')

## 결과를 어떻게 쓰나

- **검출률이 낮은데 Top-1 은 맞다** → 점수가 전반적으로 낮은 것이다. 임계값 조정을 검토한다(7번 셀).
- **Top-1 도 틀린다** → 학습 데이터 문제다. 해당 클래스의 사진을 늘리거나 혼동 쌍을 합칠지 본다.
- **특정 쌍만 반복해서 틀린다**(예: 콩나물국↔계란국) → 사람도 헷갈리는 쌍인지 보고, 그렇다면 둘 다 후보로 보여주는 UI 가 나을 수 있다.
- 이 수치는 **AI Hub 스튜디오 사진 기준**이다. 실제 사용자가 찍는 한 상 사진의 체감과는 다르다 — 그쪽은 실제 식단 사진으로 따로 본다.